# Task 11: Vision Transformer (ViT) Patch Projection and Multi-Head Self-Attention from Scratch

## Objective

Build the main components of a Vision Transformer from first principles.

The implementation includes:

- Image-to-patch conversion
- Custom linear patch projection
- Learnable `[CLS]` token
- Learnable positional embeddings
- Multi-head self-attention
- Einstein summation using `torch.einsum`
- Transformer block
- Attention-map visualization

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import math

torch.manual_seed(42)
np.random.seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"

# ------------------------------------------------------------
# Image configuration
# ------------------------------------------------------------

BATCH = 2
CHANNELS = 3
IMAGE_SIZE = 224
PATCH_SIZE = 16
EMBED_DIM = 128
NUM_HEADS = 8

NUM_PATCHES = (
    IMAGE_SIZE // PATCH_SIZE
) ** 2

print("Device:", device)
print("Number of patches:", NUM_PATCHES)
print("Patch dimension:",
      CHANNELS * PATCH_SIZE * PATCH_SIZE)

# Image Patch Extraction and Linear Projection

The image is reshaped into a sequence of flattened patches.

For:

\[
224\times224
\]

images with:

\[
16\times16
\]

patches:

\[
14\times14=196
\]

patches are produced.

A learnable linear layer then converts every flattened patch into a fixed-dimensional embedding.

In [ ]:
class PatchProjection(nn.Module):

    def __init__(
        self,
        image_size,
        patch_size,
        channels,
        embed_dim
    ):
        super().__init__()

        self.image_size = image_size
        self.patch_size = patch_size
        self.channels = channels

        self.num_patches = (
            image_size // patch_size
        ) ** 2

        self.patch_dim = (
            channels
            * patch_size
            * patch_size
        )

        # Custom linear patch projection
        self.projection = nn.Linear(
            self.patch_dim,
            embed_dim
        )

    def forward(self, x):

        B, C, H, W = x.shape
        P = self.patch_size

        # Extract non-overlapping patches
        patches = x.unfold(
            2, P, P
        ).unfold(
            3, P, P
        )

        # B, C, H/P, W/P, P, P
        patches = patches.permute(
            0, 2, 3, 1, 4, 5
        )

        # B, number_of_patches, patch_dimension
        patches = patches.reshape(
            B,
            self.num_patches,
            self.patch_dim
        )

        # Linear projection
        embeddings = self.projection(
            patches
        )

        return embeddings


patch_projection = PatchProjection(
    IMAGE_SIZE,
    PATCH_SIZE,
    CHANNELS,
    EMBED_DIM
).to(device)

images = torch.randn(
    BATCH,
    CHANNELS,
    IMAGE_SIZE,
    IMAGE_SIZE,
    device=device
)

patch_embeddings = patch_projection(
    images
)

print("Input:", images.shape)
print("Patch embeddings:",
      patch_embeddings.shape)

# Class Token and Positional Embeddings

A learnable `[CLS]` token is added at the beginning of the patch sequence.

Therefore:

\[
196\text{ patches}+1\text{ CLS}=197
\]

tokens are processed by the Transformer.

Learnable positional embeddings preserve spatial ordering information.

In [ ]:
class ViTEmbeddings(nn.Module):

    def __init__(
        self,
        num_patches,
        embed_dim
    ):
        super().__init__()

        # Learnable CLS token
        self.cls_token = nn.Parameter(
            torch.randn(1, 1, embed_dim)
        )

        # Learnable positional embedding
        self.position = nn.Parameter(
            torch.randn(
                1,
                num_patches + 1,
                embed_dim
            )
        )

    def forward(self, patch_embeddings):

        B = patch_embeddings.shape[0]

        cls = self.cls_token.expand(
            B, -1, -1
        )

        tokens = torch.cat(
            [cls, patch_embeddings],
            dim=1
        )

        return tokens + self.position


embedding_layer = ViTEmbeddings(
    NUM_PATCHES,
    EMBED_DIM
).to(device)

tokens = embedding_layer(
    patch_embeddings
)

print("Tokens:", tokens.shape)

# Multi-Head Self-Attention Using Einstein Summation

The token sequence is divided into multiple attention heads.

For \(h\) heads:

\[
d_{head}=\frac{D}{h}
\]

Queries, keys, and values are computed and reshaped as:

\[
[B,H,N,D_h]
\]

Attention scores are calculated using Einstein summation:

\[
S_{bhij}
=
\sum_d Q_{bhid}K_{bhjd}
\]

using `torch.einsum`.

The attention matrix is then:

\[
A=softmax(S)
\]

and the output is:

\[
O_{bhid}
=
\sum_j A_{bhij}V_{bhjd}
\]

In [ ]:
class MultiHeadSelfAttention(nn.Module):

    def __init__(
        self,
        embed_dim,
        num_heads
    ):
        super().__init__()

        assert embed_dim % num_heads == 0

        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = (
            embed_dim // num_heads
        )

        self.Wq = nn.Linear(
            embed_dim,
            embed_dim
        )

        self.Wk = nn.Linear(
            embed_dim,
            embed_dim
        )

        self.Wv = nn.Linear(
            embed_dim,
            embed_dim
        )

        self.output = nn.Linear(
            embed_dim,
            embed_dim
        )

    def forward(self, x):

        B, N, D = x.shape

        Q = self.Wq(x)
        K = self.Wk(x)
        V = self.Wv(x)

        # B, N, heads, head_dim
        Q = Q.reshape(
            B, N,
            self.num_heads,
            self.head_dim
        )

        K = K.reshape(
            B, N,
            self.num_heads,
            self.head_dim
        )

        V = V.reshape(
            B, N,
            self.num_heads,
            self.head_dim
        )

        # B, heads, N, head_dim
        Q = Q.permute(0, 2, 1, 3)
        K = K.permute(0, 2, 1, 3)
        V = V.permute(0, 2, 1, 3)

        # Einstein summation:
        # [B,H,N,D] x [B,H,M,D]
        # -> [B,H,N,M]
        scores = torch.einsum(
            "bhid,bhjd->bhij",
            Q,
            K
        )

        scores = scores / math.sqrt(
            self.head_dim
        )

        attention = torch.softmax(
            scores,
            dim=-1
        )

        # Attention x Values
        output = torch.einsum(
            "bhij,bhjd->bhid",
            attention,
            V
        )

        # B,H,N,D -> B,N,H,D
        output = output.permute(
            0, 2, 1, 3
        )

        output = output.reshape(
            B, N, D
        )

        output = self.output(
            output
        )

        return output, attention


attention = MultiHeadSelfAttention(
    EMBED_DIM,
    NUM_HEADS
).to(device)

attention_output, attention_maps = attention(
    tokens
)

print(
    "Attention output:",
    attention_output.shape
)

print(
    "Attention maps:",
    attention_maps.shape
)

# Vision Transformer Block

A standard Transformer block combines:

1. Layer Normalization
2. Multi-head self-attention
3. Residual connection
4. Feed-forward network
5. Second residual connection

The structure is:

\[
X'=X+MSA(LN(X))
\]

\[
Y=X'+MLP(LN(X'))
\]

In [ ]:
class ViTBlock(nn.Module):

    def __init__(
        self,
        embed_dim,
        num_heads,
        mlp_ratio=4
    ):
        super().__init__()

        self.norm1 = nn.LayerNorm(
            embed_dim
        )

        self.attention = (
            MultiHeadSelfAttention(
                embed_dim,
                num_heads
            )
        )

        self.norm2 = nn.LayerNorm(
            embed_dim
        )

        self.mlp = nn.Sequential(
            nn.Linear(
                embed_dim,
                embed_dim * mlp_ratio
            ),
            nn.GELU(),
            nn.Linear(
                embed_dim * mlp_ratio,
                embed_dim
            )
        )

    def forward(self, x):

        attn_out, attention = (
            self.attention(
                self.norm1(x)
            )
        )

        x = x + attn_out

        x = x + self.mlp(
            self.norm2(x)
        )

        return x, attention


vit_block = ViTBlock(
    EMBED_DIM,
    NUM_HEADS
).to(device)

block_output, attention_maps = vit_block(
    tokens
)

print("ViT block output:",
      block_output.shape)

print("Attention map:",
      attention_maps.shape)

In [ ]:
# ============================================================
# Full ViT forward pass
# ============================================================

patches = patch_projection(images)

tokens = embedding_layer(patches)

output, attention_maps = vit_block(
    tokens
)

# CLS token representation
cls_representation = output[:, 0, :]

print("Patch sequence:", patches.shape)
print("Token sequence:", tokens.shape)
print("Transformer output:", output.shape)
print("CLS representation:",
      cls_representation.shape)


# ============================================================
# Attention visualization
# ============================================================

# Use first image and first attention head
# CLS token attending to image patches
cls_attention = attention_maps[
    0, 0, 0, 1:
].detach().cpu().numpy()

attention_map = cls_attention.reshape(
    IMAGE_SIZE // PATCH_SIZE,
    IMAGE_SIZE // PATCH_SIZE
)

plt.figure(figsize=(6, 6))

plt.imshow(
    attention_map,
    cmap="viridis"
)

plt.colorbar(
    label="Attention"
)

plt.title(
    "ViT CLS-to-Patch Attention Map"
)

plt.xlabel("Patch Column")
plt.ylabel("Patch Row")

plt.show()

# Conclusion

A Vision Transformer pipeline was implemented from scratch using PyTorch.

The implementation demonstrated:

- Non-overlapping image patch extraction
- Custom linear patch projection
- Learnable `[CLS]` token
- Learnable positional embeddings
- Multi-head self-attention
- Einstein summation using `torch.einsum`
- Transformer residual connections
- Feed-forward network
- Attention-map extraction and visualization

For a \(224\times224\) image with \(16\times16\) patches, the image produces:

\[
14\times14=196
\]

patch tokens. After adding the `[CLS]` token, the Transformer processes:

\[
197
\]

tokens.

The attention visualization demonstrates how the `[CLS]` token attends to different spatial image patches, providing an interpretable view of the self-attention mechanism.